# Joint Kinematics: Peak Values and Range of Motion

This notebook is **Stage 1** of my joint kinematics analysis pipeline. It takes Vicon model output (joint angles per frame) and the gait-cycle event CSV from my spatiotemporal analysis notebook, segments the trial into individual gait cycles, computes per-cycle min/max/ROM for six joint angles (left/right ankle, hip, knee in the sagittal plane), and removes cycles that contain outlier values via the IQR method.

**Author**: Yeon-Joo Kang | Georgia State University | 2021–2024
**Status**: Archived. Reflects my analytical approach during PhD dissertation research.

## Input dependencies

This notebook expects two CSV files per trial:

- `{Subject}MO{Trial}.csv` — Vicon model output with joint angle columns (FRAME, LANKX/Y/Z, LFOOTX/Y/Z, LHIPX/Y/Z, LKNEX/Y/Z, and the same set for the right side)
- `{Subject}Trial{Trial}_data.csv` — Gait-cycle event dataframe produced by my spatiotemporal analysis notebook, containing left and right heel-strike frame indices

## Pipeline context

This notebook is the **first stage** in a multi-stage joint kinematics workflow:

1. **This notebook** — per-trial gait-cycle segmentation, peak values, ROM, IQR outlier removal per joint
2. `joint_kinematics_ensemble_avg.ipynb` — combine left/right and average gait cycles within each subject
3. `joint_kinematics_group_statistics.ipynb` — cross-subject outlier removal and group-level comparison statistics

## Per-joint processing pattern

For each of the six joints (L/R ankle, hip, knee), the notebook repeats the same processing pattern:

1. Segment the trial into gait cycles using LHSidx values
2. Drop columns (cycles) that are all-zero or all-NaN
3. Compute trial-level descriptive statistics (min, max, ROM)
4. Detect outliers via IQR rule applied separately to min and max columns
5. Remove outlier columns from both the descriptive dataframe and the gait-cycle dataframe
6. Visualize before/after with boxplots and line plots

I have kept all six joints in this notebook (rather than refactoring into a function) because this is a portfolio of how I worked through the analysis, not a production tool. The repetition reflects my actual workflow.

---


## 1. Setup and Data Import

Import the joint angle CSV and the gait-cycle event CSV. Column names follow the convention `{Joint}{Axis}` where Joint is one of {LANK, LFOOT, LHIP, LKNE, RANK, RFOOT, RHIP, RKNE} and Axis is X (sagittal flexion/extension), Y, or Z.

In [ ]:
# Import necessary packages
import pandas as pd
import numpy as np
import os,sys
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Check current working directory
os.getcwd()

In [ ]:
# Model Output File import by filename
Subject = input("Subject: ")
Trial = input("Trial: ")
filename = Subject + "MO"+ Trial
filename2 = Subject + "Trial"+ Trial+"_data"
df = pd.read_csv(filename +'.csv')
df2 = pd.read_csv(filename2 +'.csv')

df.head()

Set column names explicitly. Vicon exports the joint angles without headers in the first row, so I assign them here and drop the placeholder rows.

In [ ]:
# Set column names
df.columns=['FRAME','LANKX','LANKY','LANKZ','LFOOTX','LFOOTY','LFOOTZ','LHIPX','LHIPY','LHIPZ','LKNEX','LKNEY','LKNEZ',
           'RANKX','RANKY','RANKZ','RFOOTX','RFOOTY','RFOOTZ','RHIPX','RHIPY','RHIPZ','RKNEX','RKNEY','RKNEZ']
df.head()

In [ ]:
# Drop unnecessary rows
df.drop([0,1], axis=0, inplace=True)
df.head()

In [ ]:
# Reset the index
df1=df.reset_index(drop=True)
df1.head()

In [ ]:
# Make sure variables are in float dtype
flt = df1.astype(float)
flt.shape

Quick look at the gait-cycle event dataframe and the distribution of heel-strike intervals — useful for spotting trials with abnormal cadence.

In [ ]:
# Check Heel Strike distribution
sns.histplot(data=df2.iloc[1:], x="lHSinterval"),sns.histplot(data=df2.iloc[1:], x="rHSinterval")

## 2. Left Ankle

### Segment trial into gait cycles

For each left heel-strike (LHSidx), slice the joint angle dataframe between consecutive heel strikes. Each column of the resulting dataframe is one full gait cycle of the left ankle X-axis angle.

In [ ]:
### Left Ankle ################################################################################################
col_names = ['LANKX']
name_rows_dict1 = {}

for col in col_names:
    gait_cycles = []
    for idx, value in enumerate(df2['LHSidx']):
        if idx == 0:
            gait_cycles.append(0)
            gait_cycles = pd.DataFrame(gait_cycles)
            gait_cycles.columns = [idx]
        else:
            gait_cycle = flt.loc[df2.iloc[idx-1,0]:df2.iloc[idx,0],col]
            gait_cycle = gait_cycle.to_frame()
            gait_cycle = gait_cycle.reset_index(drop=True)
            gait_cycle.columns = [idx]
            gait_cycles = pd.concat([gait_cycles,gait_cycle], axis=1)

la = gait_cycles
la = la.drop([0],axis=1)
la

### Drop empty cycles

Some cycles produced from segmentation are all zero or all NaN (e.g., the very first segment before the first complete cycle). Keep only columns that contain at least one nonzero, non-NaN value.

In [ ]:
# Test: drop all zero rows and columns
import math

la_t1 = pd.DataFrame()

for col_name, col_value_list in la.items():
    for col_value in col_value_list:
        if col_value != 0 and not math.isnan(col_value):
            la_t1 = pd.concat([la_t1, la.loc[:, col_name].to_frame()], axis=1)
            break
la_t1

### Compute peak values and ROM

`describe()` gives min, max, and other statistics per cycle. I keep min and max, then compute ROM as max minus min.

In [ ]:
# Check Descriptions of left ankle data
la_t1_des = la_t1.describe()
la_t1_des

In [ ]:
# Create min, max, ROM dataframe for left ankle
lank_min = la_t1_des.loc['min',:].to_frame()
lank_max = la_t1_des.loc['max',:].to_frame()
lank = pd.concat([lank_min, lank_max], axis=1)
lank.columns =['lamin','lamax']
lank['laROM'] = lank['lamax']-lank['lamin']
lank

In [ ]:
sns.boxplot(x=lank['lamin']),sns.boxplot(x=lank['lamax'])

### Detect outliers using the IQR rule

Apply Q1−1.5×IQR and Q3+1.5×IQR thresholds to the min and max columns separately, producing four index sets of outliers. The union of these four sets gives all the gait cycles to exclude.

In [ ]:
# Outliers: Left ankle min
Q1 = np.percentile(lank['lamin'], 25, method='midpoint')
Q3 = np.percentile(lank['lamin'], 75, method='midpoint')
IQR = Q3 - Q1
upper = Q3 +1.5*IQR
lower = Q1 -1.5*IQR

idx1 = lank.index[(lank['lamin']>= upper)]
idx2 = lank.index[(lank['lamin']<= lower)]

idx1, idx2

In [ ]:
# Outliers: Left ankle max
Q1 = np.percentile(lank['lamax'], 25, method='midpoint')
Q3 = np.percentile(lank['lamax'], 75, method='midpoint')
IQR = Q3 - Q1
upper = Q3 +1.5*IQR
lower = Q1 -1.5*IQR

idx3 = lank.index[(lank['lamax']>= upper)]
idx4 = lank.index[(lank['lamax']<= lower)]

idx3, idx4

Combine all outlier indices and drop those rows from the descriptive dataframe.

In [ ]:
# Get index name of outliers as list
idxs = list(idx1)+list(idx2)+list(idx3)+list(idx4)
idxsl = set(idxs)

# Drop Outliers in left knee description data based on min,max values
lank_t=lank
lank_t = lank_t.drop(index=(idxsl))

lank_t.shape, lank.shape

In [ ]:
sns.boxplot(x=lank_t['lamin']), sns.boxplot(x=lank_t['lamax'])

In [ ]:
# Reset the index of min,max,ROM dataframe
lank_t = lank_t.reset_index(drop=True)
lank_t

Drop the same outlier columns from the per-cycle gait dataframe so the descriptive and cycle dataframes stay synchronized.

In [ ]:
# Drop Outliers in left ankle gait cycle data based on min,max values
la_tt=la_t1
la_tt = la_tt.drop(idxsl, axis=1)

la_tt.shape, la_t1.shape

In [ ]:
sns.lineplot(data=la_tt).get_legend().remove()

## 3. Right Ankle

Apply the same segment → drop empty → describe → IQR outlier → drop outliers → visualize pipeline to the right ankle.

In [ ]:
### Right Ankle ##############################################################################################
col_names = ['RANKX']
name_rows_dict1 = {}

for col in col_names:
    gait_cycles = []
    for idx, value in enumerate(df2['RHSidx']):
        if idx == 0:
            gait_cycles.append(0)
            gait_cycles = pd.DataFrame(gait_cycles)
            gait_cycles.columns = [idx]
        else:
            gait_cycle = flt.loc[df2.iloc[idx-1,8]:df2.iloc[idx,8],col]
            gait_cycle = gait_cycle.to_frame()
            gait_cycle = gait_cycle.reset_index(drop=True)
            gait_cycle.columns = [idx]
            gait_cycles = pd.concat([gait_cycles,gait_cycle], axis=1)

ra = gait_cycles
ra = ra.drop([0],axis=1)
ra

In [ ]:
# Test: drop all zero rows and columns
import math

ra_t1 = pd.DataFrame()

for col_name, col_value_list in ra.items():
    for col_value in col_value_list:
        if col_value != 0 and not math.isnan(col_value):
            ra_t1 = pd.concat([ra_t1, ra.loc[:, col_name].to_frame()], axis=1)
            break
ra_t1

In [ ]:
# Check descriptions of Right ankle data
ra_t1_des = ra_t1.describe()
ra_t1_des

In [ ]:
# Create min, max, ROM dataframe for right ankle
rank_min = ra_t1_des.loc['min',:].to_frame()
rank_max = ra_t1_des.loc['max',:].to_frame()
rank = pd.concat([rank_min, rank_max], axis=1)
rank.columns =['ramin','ramax']
rank['raROM'] = rank['ramax']-rank['ramin']
rank

In [ ]:
sns.boxplot(x=rank['ramin']),sns.boxplot(x=rank['ramax'])

In [ ]:
# Outliers: right ankle min
Q1 = np.percentile(rank['ramin'], 25, method='midpoint')
Q3 = np.percentile(rank['ramin'], 75, method='midpoint')
IQR = Q3 - Q1
upper = Q3 +1.5*IQR
lower = Q1 -1.5*IQR

idx1 = rank.index[(rank['ramin']>= upper)]
idx2 = rank.index[(rank['ramin']<= lower)]

idx1, idx2

In [ ]:
# Outliers: Right ankle max
Q1 = np.percentile(rank['ramax'], 25, method='midpoint')
Q3 = np.percentile(rank['ramax'], 75, method='midpoint')
IQR = Q3 - Q1
upper = Q3 +1.5*IQR
lower = Q1 -1.5*IQR

idx3 = rank.index[(rank['ramax']>= upper)]
idx4 = rank.index[(rank['ramax']<= lower)]

idx3, idx4

In [ ]:
# Get index name of outliers as list
idxs = list(idx1)+list(idx2)+list(idx3)+list(idx4)
idxsl = set(idxs)

# Drop Outliers in Right ankle description data based on min,max values
rank_t=rank
rank_t = rank_t.drop(index=(idxsl))

rank_t.shape, rank.shape

In [ ]:
sns.boxplot(x=rank_t['ramin']),sns.boxplot(x=rank_t['ramax'])

In [ ]:
# Reset the index of min,max,ROM dataframe
rank_t = rank_t.reset_index(drop=True)
rank_t

In [ ]:
# Drop Outliers in Right ankle gait cycle data based on min,max values
ra_tt=ra_t1
ra_tt = ra_tt.drop(idxsl, axis=1)

ra_tt.shape, ra_t1.shape

In [ ]:
sns.lineplot(data=ra_tt).get_legend().remove()

## 4. Left Hip

Same processing pattern applied to the left hip.

In [ ]:
### Left Hip ##################################################################################################
col_names = ['LHIPX']
name_rows_dict1 = {}

for col in col_names:
    gait_cycles = []
    for idx, value in enumerate(df2['LHSidx']):
        if idx == 0:
            gait_cycles.append(0)
            gait_cycles = pd.DataFrame(gait_cycles)
            gait_cycles.columns = [idx]
        else:
            gait_cycle = flt.loc[df2.iloc[idx-1,0]:df2.iloc[idx,0],col]
            gait_cycle = gait_cycle.to_frame()
            gait_cycle = gait_cycle.reset_index(drop=True)
            gait_cycle.columns = [idx]
            gait_cycles = pd.concat([gait_cycles,gait_cycle], axis=1)


lh = gait_cycles
lh = lh.drop([0],axis=1)
lh

In [ ]:
# Test: drop all zero rows and columns
import math

lh_t1 = pd.DataFrame()

for col_name, col_value_list in lh.items():
    for col_value in col_value_list:
        if col_value != 0 and not math.isnan(col_value):
            lh_t1 = pd.concat([lh_t1, lh.loc[:, col_name].to_frame()], axis=1)
            break
lh_t1

In [ ]:
# Check descriptions of Left Hip data
lh_t1_des = lh_t1.describe()
lh_t1_des

In [ ]:
# Create min, max, ROM dataframe for left hip
lhip_min = lh_t1_des.loc['min',:].to_frame()
lhip_max = lh_t1_des.loc['max',:].to_frame()
lhip = pd.concat([lhip_min, lhip_max], axis=1)
lhip.columns =['lhmin','lhmax']
lhip['lhROM'] = lhip['lhmax']-lhip['lhmin']
lhip

In [ ]:
sns.boxplot(x=lhip['lhmin']),sns.boxplot(x=lhip['lhmax'])

In [ ]:
# Outliers: left hip min
Q1 = np.percentile(lhip['lhmin'], 25, method='midpoint')
Q3 = np.percentile(lhip['lhmin'], 75, method='midpoint')
IQR = Q3 - Q1
upper = Q3 +1.5*IQR
lower = Q1 -1.5*IQR

idx1 = lhip.index[(lhip['lhmin']>= upper)]
idx2 = lhip.index[(lhip['lhmin']<= lower)]

idx1, idx2

In [ ]:
# Outliers: left hip max
Q1 = np.percentile(lhip['lhmax'], 25, method='midpoint')
Q3 = np.percentile(lhip['lhmax'], 75, method='midpoint')
IQR = Q3 - Q1
upper = Q3 +1.5*IQR
lower = Q1 -1.5*IQR

idx3 = lhip.index[(lhip['lhmax']>= upper)]
idx4 = lhip.index[(lhip['lhmax']<= lower)]

idx3, idx4

In [ ]:
# Get index name of outliers as list
idxs = list(idx1)+list(idx2)+list(idx3)+list(idx4)
idxsl = set(idxs)

# Drop Outliers in Left Hip description data based on min,max values
lhip_t=lhip
lhip_t = lhip_t.drop(index=(idxsl))

lhip_t.shape, lhip.shape

In [ ]:
sns.boxplot(x=lhip_t['lhmin']),sns.boxplot(x=lhip_t['lhmax'])

In [ ]:
# Reset the index of min,max,ROM dataframe
lhip_t = lhip_t.reset_index(drop=True)
lhip_t

In [ ]:
# Drop Outliers in Left Hip gait cycle data based on min,max values
lh_tt=lh_t1
lh_tt = lh_tt.drop(idxsl, axis=1)

lh_tt.shape, lh_t1.shape

In [ ]:
sns.lineplot(data=lh_tt).get_legend().remove()

## 5. Right Hip

Same processing pattern applied to the right hip.

In [ ]:
### Right Hip ################################################################################################
col_names = ['RHIPX']
name_rows_dict1 = {}

for col in col_names:
    gait_cycles = []
    for idx, value in enumerate(df2['RHSidx']):
        if idx == 0:
            gait_cycles.append(0)
            gait_cycles = pd.DataFrame(gait_cycles)
            gait_cycles.columns = [idx]
        else:
            gait_cycle = flt.loc[df2.iloc[idx-1,8]:df2.iloc[idx,8],col]
            gait_cycle = gait_cycle.to_frame()
            gait_cycle = gait_cycle.reset_index(drop=True)
            gait_cycle.columns = [idx]
            gait_cycles = pd.concat([gait_cycles,gait_cycle], axis=1)


rh = gait_cycles
rh = rh.drop([0],axis=1)
rh

In [ ]:
# Test: drop all zero rows and columns
import math

rh_t1 = pd.DataFrame()

for col_name, col_value_list in rh.items():
    for col_value in col_value_list:
        if col_value != 0 and not math.isnan(col_value):
            rh_t1 = pd.concat([rh_t1, rh.loc[:, col_name].to_frame()], axis=1)
            break
rh_t1

In [ ]:
# Check descriptions of Right angle data
rh_t1_des = rh_t1.describe()
rh_t1_des

In [ ]:
# Create min, max, ROM dataframe for left hip
rhip_min = rh_t1_des.loc['min',:].to_frame()
rhip_max = rh_t1_des.loc['max',:].to_frame()
rhip = pd.concat([rhip_min, rhip_max], axis=1)
rhip.columns =['rhmin','rhmax']
rhip['rhROM'] = rhip['rhmax']-rhip['rhmin']
rhip

In [ ]:
sns.boxplot(x=rhip['rhmin']),sns.boxplot(x=rhip['rhmax'])

In [ ]:
# Outliers: rigth hip min
Q1 = np.percentile(rhip['rhmin'], 25, method='midpoint')
Q3 = np.percentile(rhip['rhmin'], 75, method='midpoint')
IQR = Q3 - Q1
upper = Q3 +1.5*IQR
lower = Q1 -1.5*IQR

idx1 = rhip.index[(rhip['rhmin']>= upper)]
idx2 = rhip.index[(rhip['rhmin']<= lower)]

idx1, idx2

In [ ]:
# Outliers: rigth hip max
Q1 = np.percentile(rhip['rhmax'], 25, method='midpoint')
Q3 = np.percentile(rhip['rhmax'], 75, method='midpoint')
IQR = Q3 - Q1
upper = Q3 +1.5*IQR
lower = Q1 -1.5*IQR

idx3 = rhip.index[(rhip['rhmax']>= upper)]
idx4 = rhip.index[(rhip['rhmax']<= lower)]

idx3, idx4

In [ ]:
# Get index name of outliers as list
idxs = list(idx1)+list(idx2)+list(idx3)+list(idx4)
idxsl = set(idxs)

# Drop Outliers in right Hip description data based on min,max values
rhip_t=rhip
rhip_t = rhip_t.drop(index=(idxsl))

rhip_t.shape, rhip.shape

In [ ]:
sns.boxplot(x=rhip_t['rhmin']),sns.boxplot(x=rhip_t['rhmax'])

In [ ]:
# Reset the index of min,max,ROM dataframe
rhip_t = rhip_t.reset_index(drop=True)
rhip_t

In [ ]:
# Drop Outliers in Right Hip gait cycle data based on min,max values
rh_tt=rh_t1
rh_tt = rh_tt.drop(idxsl, axis=1)

rh_tt.shape, rh_t1.shape

In [ ]:
sns.lineplot(data=rh_tt).get_legend().remove()

## 6. Left Knee

Same processing pattern applied to the left knee.

In [ ]:
###Left Knee #################################################################################################
col_names = ['LKNEX']
name_rows_dict1 = {}

for col in col_names:
    gait_cycles = []
    for idx, value in enumerate(df2['LHSidx']):
        if idx == 0:
            gait_cycles.append(0)
            gait_cycles = pd.DataFrame(gait_cycles)
            gait_cycles.columns = [idx]
        else:
            gait_cycle = flt.loc[df2.iloc[idx-1,0]:df2.iloc[idx,0],col]
            gait_cycle = gait_cycle.to_frame()
            gait_cycle = gait_cycle.reset_index(drop=True)
            gait_cycle.columns = [idx]
            gait_cycles = pd.concat([gait_cycles,gait_cycle], axis=1)


lk = gait_cycles
lk = lk.drop([0],axis=1)
lk

In [ ]:
# Test: drop all zero rows and columns
import math

lk_t1 = pd.DataFrame()

for col_name, col_value_list in lk.items():
    for col_value in col_value_list:
        if col_value != 0 and not math.isnan(col_value):
            lk_t1 = pd.concat([lk_t1, lk.loc[:, col_name].to_frame()], axis=1)
            break
lk_t1

In [ ]:
# Check descriptions of Right angle data
lk_t1_des = lk_t1.describe()
lk_t1_des

In [ ]:
# Create min, max, ROM dataframe for left knee
lknee_min = lk_t1_des.loc['min',:].to_frame()
lknee_max = lk_t1_des.loc['max',:].to_frame()
lknee = pd.concat([lknee_min, lknee_max], axis=1)
lknee.columns =['lkmin','lkmax']
lknee['lkROM'] = lknee['lkmax']-lknee['lkmin']
lknee

In [ ]:
sns.boxplot(x=lknee['lkmin']),sns.boxplot(x=lknee['lkmax'])

In [ ]:
# Outliers: left knee min
Q1 = np.percentile(lknee['lkmin'], 25, method='midpoint')
Q3 = np.percentile(lknee['lkmin'], 75, method='midpoint')
IQR = Q3 - Q1
upper = Q3 +1.5*IQR
lower = Q1 -1.5*IQR

idx1 = lknee.index[(lknee['lkmin']>= upper)]
idx2 = lknee.index[(lknee['lkmin']<= lower)]

idx1, idx2

In [ ]:
# Outliers: left knee max
Q1 = np.percentile(lknee['lkmax'], 25, method='midpoint')
Q3 = np.percentile(lknee['lkmax'], 75, method='midpoint')
IQR = Q3 - Q1
upper = Q3 +1.5*IQR
lower = Q1 -1.5*IQR

idx3 = lknee.index[(lknee['lkmax']>= upper)]
idx4 = lknee.index[(lknee['lkmax']<= lower)]

idx3, idx4

In [ ]:
# Get index name of outliers as list
idxs = list(idx1)+list(idx2)+list(idx3)+list(idx4)
idxsl = set(idxs)

# Drop Outliers in left knee description data based on min,max values
lknee_t=lknee
lknee_t = lknee_t.drop(index=(idxsl))

lknee_t.shape, lknee.shape

In [ ]:
sns.boxplot(x=lknee_t['lkmin']),sns.boxplot(x=lknee_t['lkmax'])

In [ ]:
# Reset the index of min,max,ROM dataframe
lknee_t = lknee_t.reset_index(drop=True)
lknee_t

In [ ]:
# Drop Outliers in Left Knee gait cycle data based on min,max values
lk_tt = lk_t1
lk_tt = lk_tt.drop(idxsl, axis=1)

lk_tt.shape, lk_t1.shape

In [ ]:
sns.lineplot(data=lk_tt).get_legend().remove()

## 7. Right Knee

Same processing pattern applied to the right knee.

In [ ]:
### Right Side Knee ###########################################################################################
col_names = ['RKNEX']
name_rows_dict1 = {}

for col in col_names:
    gait_cycles = []
    for idx, value in enumerate(df2['RHSidx']):
        if idx == 0:
            gait_cycles.append(0)
            gait_cycles = pd.DataFrame(gait_cycles)
            gait_cycles.columns = [idx]
        else:
            gait_cycle = flt.loc[df2.iloc[idx-1,8]:df2.iloc[idx,8],col]
            gait_cycle = gait_cycle.to_frame()
            gait_cycle = gait_cycle.reset_index(drop=True)
            gait_cycle.columns = [idx]
            gait_cycles = pd.concat([gait_cycles,gait_cycle], axis=1)


rk = gait_cycles
rk = rk.drop([0],axis=1)
rk

In [ ]:
# Test: drop all zero rows and columns
import math

rk_t1 = pd.DataFrame()

for col_name, col_value_list in rk.items():
    for col_value in col_value_list:
        if col_value != 0 and not math.isnan(col_value):
            rk_t1 = pd.concat([rk_t1, rk.loc[:, col_name].to_frame()], axis=1)
            break
rk_t1

In [ ]:
# Check descriptions of Right angle data
rk_t1_des = rk_t1.describe()
rk_t1_des

In [ ]:
# Create min, max, ROM dataframe for left knee
rknee_min = rk_t1_des.loc['min',:].to_frame()
rknee_max = rk_t1_des.loc['max',:].to_frame()
rknee = pd.concat([rknee_min, rknee_max], axis=1)
rknee.columns =['rkmin','rkmax']
rknee['rkROM'] = rknee['rkmax']-rknee['rkmin']
rknee

In [ ]:
sns.boxplot(x=rknee['rkmin']),sns.boxplot(x=rknee['rkmax'])

In [ ]:
# Outliers: left knee min
Q1 = np.percentile(rknee['rkmin'], 25, method='midpoint')
Q3 = np.percentile(rknee['rkmin'], 75, method='midpoint')
IQR = Q3 - Q1
upper = Q3 +1.5*IQR
lower = Q1 -1.5*IQR

idx1 = rknee.index[(rknee['rkmin']>= upper)]
idx2 = rknee.index[(rknee['rkmin']<= lower)]

idx1, idx2

In [ ]:
# Outliers: left knee min
Q1 = np.percentile(rknee['rkmax'], 25, method='midpoint')
Q3 = np.percentile(rknee['rkmax'], 75, method='midpoint')
IQR = Q3 - Q1
upper = Q3 +1.5*IQR
lower = Q1 -1.5*IQR

idx3 = rknee.index[(rknee['rkmax']>= upper)]
idx4 = rknee.index[(rknee['rkmax']<= lower)]

idx3, idx4

In [ ]:
# Get index name of outliers as list
idxs = list(idx1)+list(idx2)+list(idx3)+list(idx4)
idxsl = set(idxs)

# Drop Outliers in left knee description data based on min,max values
rknee_t = rknee
rknee_t = rknee_t.drop(index=(idxsl))

rknee_t.shape, rknee.shape

In [ ]:
sns.boxplot(x=rknee_t['rkmin']),sns.boxplot(x=rknee_t['rkmax'])

In [ ]:
# Reset the index of min,max,ROM dataframe
rknee_t = rknee_t.reset_index(drop=True)
rknee_t

In [ ]:
# Drop Outliers in Left Knee gait cycle data based on min,max values
rk_tt = rk_t1
rk_tt = rk_tt.drop(idxsl, axis=1)

rk_tt.shape, rk_t1.shape

In [ ]:
sns.lineplot(data=rk_tt).get_legend().remove()

## 8. Save Results

Save the per-cycle gait dataframes (one per joint, outliers removed), then the combined descriptive dataframe (min/max/ROM for all six joints side by side). The combined dataframe is what Stage 2 (`joint_kinematics_ensemble_avg.ipynb`) uses as input.

In [ ]:
# Save gait cycle data in csv 
la_tt.to_csv(filename + '_lank.csv',  index=False)
ra_tt.to_csv(filename + '_rank.csv',  index=False)
lh_tt.to_csv(filename + '_lhip.csv',  index=False)
rh_tt.to_csv(filename + '_rhip.csv',  index=False)
lk_tt.to_csv(filename + '_lknee.csv',  index=False)
rk_tt.to_csv(filename + '_rknee.csv',  index=False)

In [ ]:
jangle = pd.concat([lank_t,rank_t,lhip_t,rhip_t,lknee_t,rknee_t], axis=1)
jangle

In [ ]:
# Save joint angle data in csv 
jangle.to_csv(filename + '_data.csv',  index=False)

Append per-trial mean and SD across all joint angle columns to a cumulative CSV for cross-trial comparison within a subject.

In [ ]:
# Get Mean and SD for joint angle

angle = list(jangle)
rows = []

for col in angle:
    df_attr = getattr(jangle, col)
    row = pd.DataFrame({col+"_Avg":[df_attr.mean()], 
                        col+"_SD":[df_attr.std()]
                       }, index = [filename])
    rows.append(row)

angle_df = pd.concat(rows, axis=1)
angle_df

In [ ]:
#Save Mean and SD for joint angle
if not os.path.exists('jangle1.csv'):
    angle_df.to_csv('jangle1.csv', index=filename, mode='w')
else:
    angle_df.to_csv('jangle1.csv', index=filename, mode='a')